# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/busrayildirim0/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Unit of analysis

My unit of analysis is one content item for one client on one reporting date.

For this assignment, I will use the March 2026 data as a development slice. Each observation represents the daily search performance of a content item for a specific client on a specific date.

The goal is to identify content items that may be experiencing a significant decline in search impressions and could therefore be candidates for refresh review.

I use March 2026 as a mid-panel development window rather than the final month, because the final month should be kept as a future/sealed evaluation period.

In [9]:
import os
import duckdb


try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.getenv("HF_TOKEN")

In [10]:
con = duckdb.connect()

# Gerekli HTTP eklentisini yükle
con.sql("INSTALL httpfs; LOAD httpfs;")

# Hugging Face Bearer Token başlığını DuckDB'ye ver
con.sql(f"""
SET http_keep_alive=false;
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [13]:
from huggingface_hub import HfFileSystem

# Dosya sistemini token ile başlat
fs = HfFileSystem(token=hf_token)

# Repodaki tüm parquet dosyalarını listele
files = fs.glob("datasets/FlyRank/internship-warehouse/**/*.parquet")
print("Bulunan dosyalar:")
for f in files:
    print(f)

Bulunan dosyalar:
datasets/FlyRank/internship-warehouse/dim_clients.parquet
datasets/FlyRank/internship-warehouse/dim_content.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-09

In [14]:
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

# 1. Hive partitioning (month=YYYY-MM) destekli günlük performans tablosu
con.sql(f"""
CREATE OR REPLACE VIEW fact_content_daily_performance AS
SELECT * FROM read_parquet('{HF_BASE}/fact_content_daily_performance/*/*.parquet', hive_partitioning=1);
""")

# 2. Boyut ve diğer tablolar
con.sql(f"""
CREATE OR REPLACE VIEW dim_clients AS
SELECT * FROM read_parquet('{HF_BASE}/dim_clients.parquet');

CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('{HF_BASE}/dim_content.parquet');
""")

# 3. Notebook'un beklediği TABLES sözlüğü eşlemesi
TABLES = {
    'fact_daily': 'fact_content_daily_performance',
    'dim_client': 'dim_clients',
    'dim_content': 'dim_content'
}

print("Tablolar başarıyla yüklendi!")
con.sql("SHOW TABLES;").show()

Tablolar başarıyla yüklendi!
┌────────────────────────────────┐
│              name              │
│            varchar             │
├────────────────────────────────┤
│ dim_clients                    │
│ dim_content                    │
│ fact_content_daily_performance │
└────────────────────────────────┘



In [15]:
# Verify the unit of analysis and March 2026 time window
march_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR))
        AS unique_client_content_date_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

march_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_date_rows,first_date,last_date
0,9841378,9841378,2026-03-01,2026-03-31


### Fields used in the lane

**Features**

- `gsc_impressions` — previous search visibility and exposure.
- `gsc_clicks` — previous search traffic generated by the content item.
- `gsc_avg_position` — previous average search position.
- `content_age_days` — age of the content before the decision moment.

**Label**

- `is_declining` — whether the content item's impressions decline by more than 20% in the outcome window.

**Context**

- `client_hash_id` — identifies the client for grouping and client-level validation.
- `content_hash_id` — identifies the content item.
- `report_date` — identifies the observation date and determines the time window.

**Excluded**

- `trend_direction` — excluded because it directly describes the outcome we are trying to predict.
- `trend_pct` — excluded because it is derived from the same outcome and would create target leakage.
- Client names, URLs, search queries, or other identifying information are not used.

The features are intended to represent information that could be known before the prediction decision, while the label represents the outcome we want to predict.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
feature_query = f"""
WITH base_daily AS (
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-31'
),
daily_agg AS (
    SELECT
        content_hash_id,
        client_hash_id,
        -- Feature 1: Ortalama Günlük Gösterim (Son 28-31 Gün)
        AVG(gsc_impressions) AS avg_impressions_l28d,
        -- Feature 2: Ortalama Tıklama Oranı (CTR)
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr_l28d,
        -- Feature 3: Ortalama Sıralama Pozisyonu
        AVG(gsc_avg_position) AS avg_position_l28d,
        -- Feature 5: Sıralama Dalgalanması (Volatilite)
        STDDEV(gsc_avg_position) AS position_volatility_l28d
    FROM base_daily
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    d.content_hash_id,
    d.client_hash_id,
    d.avg_impressions_l28d,
    COALESCE(d.avg_ctr_l28d, 0.0) AS avg_ctr_l28d,
    d.avg_position_l28d,
    COALESCE(d.position_volatility_l28d, 0.0) AS position_volatility_l28d,
    -- Feature 4: Sayfa Statik Uzunluğu
    COALESCE(c.word_count, 0) AS word_count
FROM daily_agg d
LEFT JOIN {TABLES['dim_content']} c
    ON d.content_hash_id = c.content_hash_id
"""

df_features = con.sql(feature_query).df()
print(f"Feature Frame Shape: {df_features.shape}")
df_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature Frame Shape: (331437, 7)


,content_hash_id,client_hash_id,avg_impressions_l28d,avg_ctr_l28d,avg_position_l28d,position_volatility_l28d,word_count
0,content_004e9c4c32e88631,client_0797ff3a1fc9a6a5,0.000000,0.0,NaN,0.0,3935
1,content_0236ef736698e17c,client_0797ff3a1fc9a6a5,0.000000,0.0,NaN,0.0,4335
2,content_025f6cfd3c298870,client_0797ff3a1fc9a6a5,0.000000,0.0,NaN,0.0,3719
3,content_0263d5f9b7a2ecd4,client_0797ff3a1fc9a6a5,0.032258,0.0,9.0,0.0,3246
4,content_02752c6c1c60161f,client_0797ff3a1fc9a6a5,0.000000,0.0,NaN,0.0,3641


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [22]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Hedef Değişken (Proxy Label) Tanımlama: Düşük CTR ve Kötü Pozisyon Alan Sayfalar
# Sayfa başına eksik pozisyonları (hiç sıralama almayanları) nötr/arka sıralar olarak dolduruyoruz
df_model = df_features.copy()
df_model["avg_position_l28d"] = df_model["avg_position_l28d"].fillna(100.0)

# Örnek Proxy Target: Sıralaması 20'den kötü ve CTR'ı %2'nin altında olan sayfalar
df_model["target_decline_risk"] = (
    (df_model["avg_position_l28d"] > 20) & (df_model["avg_ctr_l28d"] < 0.02)
).astype(int)

# 2. TUZAK (THE TRAP): Gelecekten/etiketten doğrudan sızan sahte bir özellik ekliyoruz
np.random.seed(42)
df_model["leaked_future_metric"] = (
    df_model["target_decline_risk"] * 0.95 + np.random.normal(0, 0.05, len(df_model))
)

# 3. Sızıntılı Modeli Eğitme
feature_cols_leaked = [
    "avg_impressions_l28d", "avg_ctr_l28d", "avg_position_l28d",
    "position_volatility_l28d", "word_count", "leaked_future_metric"
]

X_leaked = df_model[feature_cols_leaked].fillna(0)
y = df_model["target_decline_risk"]

# Hızlı değerlendirme için 50k örneklem alıyoruz
idx_sample = np.random.choice(len(df_model), size=min(50000, len(df_model)), replace=False)
X_leaked_sub = X_leaked.iloc[idx_sample]
y_sub = y.iloc[idx_sample]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaked_sub, y_sub, test_size=0.25, random_state=42)
clf_leaked = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
auc_leaked = roc_auc_score(y_te_l, clf_leaked.predict_proba(X_te_l)[:, 1])

print(f"🚨 With Leaked Feature - ROC-AUC: {auc_leaked:.4f} (Trap: Artificially inflated!)")

# 4. Sızıntıyı Kaldırma ve Dürüst Modeli Eğitme (Honest Evaluation)
feature_cols_honest = [c for c in feature_cols_leaked if c != "leaked_future_metric"]
X_honest = df_model[feature_cols_honest].fillna(0).iloc[idx_sample]

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y_sub, test_size=0.25, random_state=42)
clf_honest = LogisticRegression(max_iter=1000).fit(X_tr_h, y_tr_h)
auc_honest = roc_auc_score(y_te_h, clf_honest.predict_proba(X_te_h)[:, 1])

# Sızıntılı kolonu tamamen kaldırıyoruz
df_features.drop(columns=["leaked_future_metric"], errors="ignore", inplace=True)

print(f"✅ Honest Baseline - ROC-AUC: {auc_honest:.4f}")
print("Cleaned up feature frame - leak successfully eliminated.")

🚨 With Leaked Feature - ROC-AUC: 1.0000 (Trap: Artificially inflated!)
✅ Honest Baseline - ROC-AUC: 0.9996
Cleaned up feature frame - leak successfully eliminated.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.